In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [2]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [2]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [3]:
# Modelde kullanmak istediğimiz bağımsız değişkenler (features)
features = [
    "wait_time",
    "expected_wait_time",
    "delay_vs_expected",
    "distance_seller_customer",
    "price",
    "freight_value"
]

🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [4]:
# 1. Sadece belirlediğimiz feature'ları alalım
X = orders[features].copy()

# 2. VIF hesaplarken hata almamak için boş (NaN) satırları temizleyelim
X = X.dropna()

# 3. Veriyi standardize edelim: (Değer - Ortalama) / Standart Sapma
X_standardized = (X - X.mean()) / X.std()

# İşlemin başarılı olduğunu görmek için ilk 5 satıra bakalım
X_standardized.head()

,wait_time,expected_wait_time,delay_vs_expected,distance_seller_customer,price,freight_value
0,-0.431192,-0.934806,-0.161781,-0.979475,-0.513802,-0.652038
1,0.134174,-0.524871,-0.161781,0.429743,-0.086640,0.000467
2,-0.329907,0.330878,-0.161781,-0.145495,0.111748,-0.164053
3,0.073540,0.279445,-0.161781,2.054621,-0.441525,0.206815
4,-1.019535,-1.326297,-0.161781,-0.959115,-0.562388,-0.652038


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [6]:
# Veri tiplerini ve boş değer olup olmadığını kontrol edelim
print(X_standardized.info())
print("\nEksik veya Sonsuz Değer Kontrolü:")
print(X_standardized.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 95872 entries, 0 to 95879
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   wait_time                 95872 non-null  float64
 1   expected_wait_time        95872 non-null  float64
 2   delay_vs_expected         95872 non-null  float64
 3   distance_seller_customer  95872 non-null  float64
 4   price                     95872 non-null  float64
 5   freight_value             95872 non-null  float64
dtypes: float64(6)
memory usage: 5.1 MB
None

Eksik veya Sonsuz Değer Kontrolü:
wait_time                   0
expected_wait_time          0
delay_vs_expected           0
distance_seller_customer    0
price                       0
freight_value               0
dtype: int64


In [7]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF hesaplamasının düzgün çalışması için görünmez bir 'sabit' (constant) sütunu ekliyoruz
X_vif = sm.add_constant(X_standardized)

# Sonuçları tablo halinde görmek için boş bir DataFrame oluşturalım
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns

# Her bir sütun (feature) için VIF değerini hesaplayalım
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

# 'const' (sabit) satırını gizleyip, feature'larımızı VIF değerine göre büyükten küçüğe sıralayalım
vif_data = vif_data[vif_data['Feature'] != 'const']
vif_data.sort_values(by="VIF", ascending=False)

,Feature,VIF
1,wait_time,3.045437
3,delay_vs_expected,2.443827
2,expected_wait_time,1.598276
4,distance_seller_customer,1.570176
6,freight_value,1.341239
5,price,1.206345


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [9]:
import statsmodels.formula.api as smf  # Python'a smf'yi tanıttığımız satır

# 1. Öncelikle hedef değişkenlerimizi (0 ve 1'lerden oluşan) veri setimize ekleyelim
orders['dim_is_one_star'] = (orders['review_score'] == 1).astype(int)
orders['dim_is_five_star'] = (orders['review_score'] == 5).astype(int)

# 2. Formülü dinamik oluşturmak için daha önce tanımladığımız features listesini kullanalım
formula_one = "dim_is_one_star ~ " + " + ".join(features)

# 3. Logit 1 modelini kuralım ve eğitelim (fit edelim)
logit_one = smf.logit(formula=formula_one, data=orders).fit()

# 4. Sonuçların özet tablosunu görelim
logit_one.summary()

Optimization terminated successfully.
         Current function value: 0.280000
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95865
Method:                           MLE   Df Model:                            6
Date:                Mon, 23 Feb 2026   Pseudo R-squ.:                  0.1247
Time:                        12:57:16   Log-Likelihood:                -26844.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -2.9984      0.036    -82.285      0.000      -3.070      -2.927
wait_time                    0.0798      0.002     39.113      0.000       0.076       0.084
expected_wait_time          -0.0199      0.002    -10.753      0.000      -0.024      -0.016
delay_vs_expected            0.0388      0.004      8.923      0.000       0.030       0.047
distance_seller_customer    -0.0003   2.36e-05    -12.889      0.000      -0.000      -0.000
price                        0.0002   5.16e-05      3.047      0.002    5.61e-05       0.000
freight_value                0.0076      0.001     14.983      0.000       0.007       0.009
============================================================================================
"""

`Logit 5️⃣`

In [10]:
# 1. 5 yıldız için formülümüzü oluşturalım
formula_five = "dim_is_five_star ~ " + " + ".join(features)

# 2. Logit 5 modelini kuralım ve eğitelim (fit edelim)
logit_five = smf.logit(formula=formula_five, data=orders).fit()

# 3. Sonuçların özet tablosunu görelim
logit_five.summary()

Optimization terminated successfully.
         Current function value: 0.641066
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       dim_is_five_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95865
Method:                           MLE   Df Model:                            6
Date:                Mon, 23 Feb 2026   Pseudo R-squ.:                 0.05179
Time:                        12:59:10   Log-Likelihood:                -61460.
converged:                       True   LL-Null:                       -64817.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    0.9235      0.021     44.313      0.000       0.883       0.964
wait_time                   -0.0565      0.001    -42.292      0.000      -0.059      -0.054
expected_wait_time           0.0075      0.001      7.477      0.000       0.006       0.009
delay_vs_expected           -0.0845      0.005    -16.278      0.000      -0.095      -0.074
distance_seller_customer     0.0002   1.45e-05     10.745      0.000       0.000       0.000
price                        0.0001    3.7e-05      4.023      0.000    7.64e-05       0.000
freight_value               -0.0046      0.000    -11.379      0.000      -0.005      -0.004
============================================================================================
"""

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [12]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [13]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/data-logit/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.03s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!